- RNN擅长处理序列数据，如文本、语音、视频等， 但不擅长处理长序列数据，因为梯度消失问题
- 一些历史：
    - 2010年左右，机器学习的工作，不同领域之间的术语、技术还有很大的差异
    - 2012年，神经网络开始被广泛应用于语音识别、图像识别、翻译、自然语言处理等任务，建模工具包（如TensorFlow、PyTorch等）也开始出现
    - 2017年，Transformer模型首次提出，彻底改变了自然语言处理（NLP）的方向，成为了NLP领域的标准模型，同时也开启了深度学习在NLP领域的广泛应用
    - 2018年，BERT模型首次提出，引入了预训练和微调的概念，使得NLP任务的性能得到了显著提升

# Attention basically

In [ ]:
import numpy as np

class Node:
    def __init__(self):
        # the vector stored at this node
        self.data = np.random.randn(20)
        # weights governing how this node interacts with other nodes
        self.wkey = np.random.randn(20, 20)
        self.wquery = np.random.randn(20, 20)
        self.wvalue = np.random.randn(20, 20)

    def key(self):
        # what do I have?
        return self.wkey @ self.data

    def query(self):
        # what am I looking for?
        return self.wquery @ self.data

    def value(self):
        # what do I publicly reveal/broadcast to others?
        return self.wvalue @ self.data


class Graph:
    def __init__(self):
        # make 10 nodes
        self.nodes = [Node() for _ in range(10)]
        # make 40 edges
        randi = lambda: np.random.randint(len(self.nodes))
        self.edges = [[randi(), randi()] for _ in range(40)]

    # Query： One of the things that I'm looking for；每个节点都有一个查询向量q，用于表示该节点正在寻找的信息
    # Key： Where the thing that I have；每个节点都有一个键向量k，用于表示该节点持有的信息
    # Value： The things that I will communicate to others；每个节点都有一个值向量v，用于表示该节点公开广播的信息
    def run(self):
        updates = []
        for i, n in enumerate(self.nodes):
            # what is this node looking for?
            q = n.query()

            # find all edges that are input to this node
            inputs = [self.nodes[ifrom] for (ifrom, ito) in self.edges if ito == i]
            if len(inputs) == 0:
                continue  # ignore
            # gather their keys, i.e. what they hold
            keys = [m.key() for m in inputs]
            # calculate the compatibilities; 值越大，说明q和k越相似
            scores = [k.dot(q) for k in keys]
            # softmax them so they sum to 1； 归一化，将值转换为概率分布
            scores = np.exp(scores)
            scores = scores / np.sum(scores)
            # gather the appropriate values with a weighted sum； 加权求和，将值根据相似度进行加权
            values = [m.value() for m in inputs]
            update = sum([s * v for s, v in zip(scores, values)])
            updates.append(update)

        for n, u in zip(self.nodes, updates):
            n.data = n.data + u  # residual connection； 残差连接，将输入和输出相加，避免梯度消失问题


g = Graph()
g.run()
print("Graph run complete. Sample node data:", g.nodes[0].data[:5])


Graph run complete. Sample node data: [-4.97660155  2.54072992 -1.36428026  4.18911054  0.34817846]
